# 第 6 讲：Benchmark、Profiling 与 Kernel

这是 CS336 Lecture 06 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_06.py`
- 官方形式：executable lecture
- 英文标题：Benchmarking, Profiling, And Kernels

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲从 GPU 和性能直觉进入实际优化：怎么 benchmark、profile，什么时候写 Triton kernel，以及 elementwise、softmax、row sum、matmul tiling 的模式。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

先复习 GPU 编程模型，再学习 benchmark/profile，最后看 Triton kernel 的几类模式。


In [1]:
lecture = 6
title = 'Benchmarking, Profiling, And Kernels'
source = 'external/cs336-lectures/lecture_06.py'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 6: Benchmarking, Profiling, And Kernels
source: external/cs336-lectures/lecture_06.py


## 2. Benchmark

benchmark 衡量一个操作在给定输入规模下跑多快。要注意 warmup、同步和输入规模。


## 3. Profiling

profiling 告诉你时间花在哪里：kernel launch、memory copy、matmul、elementwise、fusion 等。


## 4. Kernel fusion

把多个 elementwise 操作合并，减少中间 tensor 和 HBM 往返。


## 5. Triton 思维

按 program/block 处理一块数据，读入 SRAM/shared memory，计算，再写回。


## 6. Tiling

matmul 和 softmax 这类操作需要分块，以便复用数据并控制内存流量。


## 动手实验 1：Python 层 benchmark 的基本形状

先直接运行，再改一个数字或字符串。


In [2]:
import math
import timeit

def gelu_scalar(x):
    return 0.5 * x * (1 + math.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

xs = [i / 1000 for i in range(10000)]
t = timeit.timeit(lambda: [gelu_scalar(x) for x in xs], number=20)
print("seconds:", t)


seconds: 0.04209775000344962


## 动手实验 2：Fusion 的内存直觉

先直接运行，再改一个数字或字符串。


In [3]:
n = 1_000_000
bytes_per_value = 4
unfused_reads_writes = 6 * n * bytes_per_value
fused_reads_writes = 2 * n * bytes_per_value
print("unfused MB:", unfused_reads_writes / 1e6)
print("fused MB:", fused_reads_writes / 1e6)


unfused MB: 24.0
fused MB: 8.0


## 动手实验 3：Tiling matmul 的块数

先直接运行，再改一个数字或字符串。


In [4]:
import math

M = N = K = 4096
block = 128
tiles_mn = math.ceil(M / block) * math.ceil(N / block)
tiles_k = math.ceil(K / block)
print("output tiles:", tiles_mn)
print("K tiles per output tile:", tiles_k)


output tiles: 1024
K tiles per output tile: 32


## 今日检查点

1. 能区分 benchmark 和 profiling。
2. 能解释 kernel fusion 为什么能减少内存流量。
3. 能说出 Triton block/program 大致对应什么。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
